In [66]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# 10-K Narrative Section Extraction — MD&A, Properties & Risk Factors

Pulls the **full text** of three 10-K sections from SEC EDGAR for the last 15 years and
stores one parquet per company in `bond_shared/data/raw/text/`

| Section | 10-K item | Column value |
|---|---|---|
| Management's Discussion & Analysis | Item 7 | `mda` |
| Properties | Item 2 | `properties` |
| Risk Factors | Item 1A | `risk_factors` |

**Steps**
1. Build the company universe (ticker + **industry**)
2. Attach CIK / company name from SEC's official ticker map
3. Keep only companies with **5+ fiscal years of 10-K history** — this drops foreign private
   issuers and recent IPOs in one rule — then count what is left
4. Extract Item 1A / Item 2 / Item 7 full text from every 10-K in the window
5. Pick the companies to process
6. Write one parquet per company into `data/raw/text/`
7. Read the parquet files back and sanity-check them

> **Where files are written.** The notebook writes to the Dropbox path below when it exists
> (i.e. when you run this on your Mac). On Colab that path does not exist, so it falls back to
> a local `./bond_shared/data/raw/text` folder and tells you so — mount Drive or copy the files
> across afterwards.

## 0. Setup

In [67]:
# Install anything missing (no-op when the package is already there).
import importlib.util, subprocess, sys

REQUIRED = [
    ("edgartools", "edgar"),
    ("pyarrow", "pyarrow"),
    ("lxml", "lxml"),
    ("html5lib", "html5lib"),
    ("beautifulsoup4", "bs4"),
]
missing = [pkg for pkg, mod in REQUIRED if importlib.util.find_spec(mod) is None]
if missing:
    print("Installing:", ", ".join(missing))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    print("All required packages already installed.")

All required packages already installed.


In [68]:
import os
import re
import time
import threading
import warnings
from io import StringIO
from urllib.parse import urlparse
from concurrent.futures import ThreadPoolExecutor

import requests
import pandas as pd
from bs4 import BeautifulSoup
from edgar import Company, set_identity

warnings.simplefilter(action="ignore", category=FutureWarning)

# ---- Configuration -------------------------------------------------------
IDENTITY = "Feriha Ozturk feriha.ozturk34@stu-mail.ccny.cuny.edu"
set_identity(IDENTITY)

SEC_HEADERS = {"User-Agent": IDENTITY, "Accept-Encoding": "gzip, deflate"}
WEB_HEADERS = {"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7)"}

YEARS_BACK = 15          # how far back to pull 10-K filings
MIN_10K_YEARS = 5        # drop companies with fewer than this many years of 10-K history
N_COMPANIES = len(universe) # how many companies to extract + store in this run



START_YEAR = pd.Timestamp.today().year - YEARS_BACK
print(f"10-K window: filings from {START_YEAR} onward; "
      f"companies need >= {MIN_10K_YEARS} fiscal years in that window.")

# Narrative text lives in data/raw/text/
DROPBOX_RAW = ("/Users/feriha/Library/CloudStorage/Dropbox-CityCollege/Feriha Ozturk/bond_shared/data/raw/text")
if not os.path.isdir(os.path.dirname(DROPBOX_RAW)):
    DROPBOX_RAW = os.path.abspath("./bond_shared/data/raw/text")
    print("Dropbox path not visible here — falling back to a local folder.")
os.makedirs(DROPBOX_RAW, exist_ok=True)
print("Writing to:", DROPBOX_RAW)

pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 60)

10-K window: filings from 2011 onward; companies need >= 5 fiscal years in that window.
Dropbox path not visible here — falling back to a local folder.
Writing to: /content/bond_shared/data/raw/text


## 1. Company universe — ticker + industry


The industry name is taken from the URL so every row carries the industry it came from.
Duplicates are dropped keeping the **first** occurrence, which preserves the market-cap
ordering of the source pages.

In [69]:
INDUSTRY_URLS = [
    "https://stockanalysis.com/stocks/industry/biotechnology/",
    "https://stockanalysis.com/stocks/industry/biotechnology/?page=2",
    "https://stockanalysis.com/stocks/industry/semiconductors/",
    "https://stockanalysis.com/stocks/industry/internet-content-and-information/",
    "https://stockanalysis.com/stocks/industry/software-infrastructure/",
]


def extract_industry(url):
    """'.../industry/software-infrastructure/' -> 'Software Infrastructure'."""
    parts = urlparse(url).path.strip("/").split("/")
    return parts[parts.index("industry") + 1].replace("-", " ").title()


def scrape_industry_table(urls=INDUSTRY_URLS):
    rows = []
    for url in urls:
        industry = extract_industry(url)
        resp = requests.get(url, headers=WEB_HEADERS, timeout=60)
        resp.raise_for_status()
        table = pd.read_html(StringIO(resp.text), keep_default_na=False)[0]
        for _, r in table.iterrows():
            sym = str(r["Symbol"]).strip()
            if not sym or sym.lower() == "nan":
                continue
            rows.append({
                "ticker": sym,
                "company_name_web": str(r.get("Company Name", "")).strip(),
                "industry": industry,
                "market_cap": str(r.get("Market Cap", "")).strip(),
            })
        print(f"  {industry:<34} {len(table):>4} rows from {url}")
        time.sleep(0.4)

    df = pd.DataFrame(rows).drop_duplicates(subset="ticker", keep="first")
    return df.reset_index(drop=True)


ticker_df = scrape_industry_table()
print(f"\nUnique companies scraped: {len(ticker_df)}")

  Biotechnology                       500 rows from https://stockanalysis.com/stocks/industry/biotechnology/
  Biotechnology                        88 rows from https://stockanalysis.com/stocks/industry/biotechnology/?page=2
  Semiconductors                       70 rows from https://stockanalysis.com/stocks/industry/semiconductors/
  Internet Content And Information     70 rows from https://stockanalysis.com/stocks/industry/internet-content-and-information/
  Software Infrastructure             177 rows from https://stockanalysis.com/stocks/industry/software-infrastructure/

Unique companies scraped: 905


In [70]:
print("Companies per industry:")
print(ticker_df["industry"].value_counts().to_string())

Companies per industry:
industry
Biotechnology                       588
Software Infrastructure             177
Semiconductors                       70
Internet Content And Information     70


### All companies with their industry

In [71]:
# Show every company, not a truncated preview.
with pd.option_context("display.max_rows", None):
    display(ticker_df[["ticker", "company_name_web", "industry", "market_cap"]])

,ticker,company_name_web,industry,market_cap
0,VRTX,Vertex Pharmaceuticals Incorporated,Biotechnology,133.37B
1,MRNA,"Moderna, Inc.",Biotechnology,79.40B
2,REGN,"Regeneron Pharmaceuticals, Inc.",Biotechnology,78.69B
3,ARGX,argenx SE,Biotechnology,60.24B
4,RVMD,"Revolution Medicines, Inc.",Biotechnology,43.93B
5,ONC,BeOne Medicines AG,Biotechnology,41.03B
6,ALNY,"Alnylam Pharmaceuticals, Inc.",Biotechnology,34.25B
7,RPRX,Royalty Pharma plc,Biotechnology,33.47B
8,ROIV,Roivant Sciences Ltd.,Biotechnology,25.86B
9,INSM,Insmed Incorporated,Biotechnology,25.73B


## 2. Attach CIK and official SEC company name

`company_tickers.json` is SEC's own ticker → CIK map, so this is the authoritative link
between a ticker and its EDGAR filings.

In [72]:
def load_ticker_cik_map():
    r = requests.get("https://www.sec.gov/files/company_tickers.json",
                     headers=SEC_HEADERS, timeout=60)
    r.raise_for_status()
    return {v["ticker"].upper().strip(): (int(v["cik_str"]), v["title"])
            for v in r.json().values()}


ticker_to_cik = load_ticker_cik_map()
print(f"SEC ticker map entries: {len(ticker_to_cik):,}")

companies = ticker_df.copy()
companies["cik"] = companies["ticker"].str.upper().map(
    lambda t: ticker_to_cik.get(t, (None, None))[0])
companies["company_name"] = companies.apply(
    lambda r: ticker_to_cik.get(r["ticker"].upper(), (None, None))[1] or r["company_name_web"],
    axis=1)

print(f"Matched to a CIK: {companies['cik'].notna().sum()} / {len(companies)}")

SEC ticker map entries: 10,428
Matched to a CIK: 905 / 905


## 3. Build the usable universe

Each company's SEC *submissions* record carries its jurisdiction of incorporation, its
business address and its full filing history. These signals are computed from it:

| Column | Meaning |
|---|---|
| `n_10k_years` | distinct **fiscal years** covered by exact 10-K filings inside the 15-year window |
| `has_10k` | `n_10k_years > 0` |
| `is_fpi` | files foreign-private-issuer forms: 20-F, 40-F, 6-K, 20FR12B, 40FR12B or F-series |
| `foreign_incorporation` | `stateOfIncorporation` is not a US state/territory code (EDGAR uses codes like `E9` = Cayman Islands, `L3` = Israel, `P7` = Netherlands) |
| `foreign_address` | headquarters address is outside the US |

**A company is kept only if `n_10k_years >= 5`.** One rule covers both cuts: a foreign
private issuer has no 10-K so it scores 0, and a 2023 IPO scores 2 or 3. Both fail. This is
the condition the whole notebook depends on, since a company without 10-Ks has no
Item 1A / 2 / 7 to extract and a company with two of them is too short for panel work.

Fiscal years come from `reportDate` so they line up with the `fiscal_year` column the
extractor writes, `10-K/A` amendments are excluded so they cannot inflate a year count, and
only filings **inside the 15-year window** count — a company that filed 10-Ks in the 1990s
and then stopped scores 0, correctly.

Foreign *domicile* does **not** drop a company. Plenty of firms are incorporated in Ireland,
Bermuda, Switzerland, Cayman or British Columbia yet file ordinary 10-Ks — Jazz, Royalty
Pharma, Roivant, CRISPR, AbCellera, Zai Lab, Biohaven — and those 10-Ks are exactly as
extractable as a Delaware filer's.

> **History is counted against the full filing record.** The `recent` block of a submissions
> record holds only the latest ~1,000 filings, so a prolific filer's older 10-Ks can sit in
> paged files. Any company that looks short on history has those pages fetched and merged
> before it is dropped. Most companies have no extra pages, so this costs almost nothing.

> SEC asks for **no more than 10 requests/second**. The fetch below is throttled to ~8/s and
> retries anything that comes back empty — a rate-limited response must not be mistaken for
> "this company has no SEC record".

In [73]:
# EDGAR uses 2-letter codes for US states/territories and alphanumeric codes for elsewhere.
US_STATE_CODES = {
    "AL", "AK", "AZ", "AR", "CA", "CO", "CT", "DE", "DC", "FL", "GA", "HI",
    "ID", "IL", "IN", "IA", "KS", "KY", "LA", "ME", "MD", "MA", "MI", "MN",
    "MS", "MO", "MT", "NE", "NV", "NH", "NJ", "NM", "NY", "NC", "ND", "OH",
    "OK", "OR", "PA", "RI", "SC", "SD", "TN", "TX", "UT", "VT", "VA", "WA",
    "WV", "WI", "WY",
    "AS", "GU", "PR", "VI", "MP",   # US territories
    "X1",                            # EDGAR's "United States" catch-all
}

MIN_INTERVAL = 0.13          # ~7.7 requests/second
_throttle_lock = threading.Lock()
_last_call = [0.0]


def _throttle():
    """Global rate limiter shared by every SEC request in this notebook."""
    with _throttle_lock:
        wait = MIN_INTERVAL - (time.monotonic() - _last_call[0])
        if wait > 0:
            time.sleep(wait)
        _last_call[0] = time.monotonic()


def fetch_submission_meta(cik, session=None, retries=4):
    get = (session or requests).get
    url = f"https://data.sec.gov/submissions/CIK{int(cik):010d}.json"
    for attempt in range(retries):
        _throttle()
        try:
            r = get(url, headers=SEC_HEADERS, timeout=60)
            if r.status_code == 200:
                return r.json()
            if r.status_code == 404:
                return None
            time.sleep(1.0 + 2 * attempt)      # 403 = rate limited, back off
        except Exception:
            time.sleep(1.0 + 2 * attempt)
    return None


def tenk_years_from_block(block):
    """
    Distinct *fiscal* years covered by exact 10-K filings inside the window.

    `block` is a submissions `recent`-shaped dict of parallel arrays. Amendments are
    excluded (they duplicate a year), and the fiscal year comes from reportDate so it
    matches the `fiscal_year` column the extractor writes.
    """
    forms = block.get("form", []) or []
    filed = block.get("filingDate", []) or []
    reported = block.get("reportDate", []) or []

    years = set()
    for i, form in enumerate(forms):
        if form != "10-K":
            continue
        fdate = filed[i] if i < len(filed) else ""
        if not fdate[:4].isdigit() or int(fdate[:4]) < START_YEAR:
            continue
        rdate = reported[i] if i < len(reported) else ""
        years.add(int(rdate[:4]) if rdate[:4].isdigit() else int(fdate[:4]))
    return years


def classify_issuer(meta):
    """Turn a submissions JSON blob into foreign-issuer and filing-history signals."""
    if not meta:
        return {"sec_company_name": None, "entity_type": None,
                "state_of_incorporation": None, "incorporation_desc": None,
                "business_country": None, "business_country_desc": None,
                "has_10k": False, "has_20f_40f": False, "is_fpi": False,
                "n_10k_years": 0, "first_10k_fy": None, "last_10k_fy": None,
                "foreign_incorporation": False, "foreign_address": False,
                "domicile_note": "no SEC submissions record"}

    soi = (meta.get("stateOfIncorporation") or "").strip().upper()
    soi_desc = (meta.get("stateOfIncorporationDescription") or "").strip()
    addr = (meta.get("addresses") or {}).get("business") or {}
    biz = (addr.get("stateOrCountry") or "").strip().upper()
    biz_desc = (addr.get("stateOrCountryDescription") or "").strip()

    recent = meta.get("filings", {}).get("recent", {})
    forms = set(recent.get("form", []))
    tenk_years = tenk_years_from_block(recent)
    has_10k = bool(tenk_years)
    has_20f_40f = any(f.startswith(("20-F", "40-F")) for f in forms)

    # Foreign private issuers file 6-K instead of 8-K and register on 20FR12B / 40FR12B /
    # F-series. Matching only "20-F"/"40-F" misses an FPI that has registered but not yet
    # filed its first annual report (e.g. SK hynix, Clinuvel, Vicore, Gix, Bitzero).
    is_fpi = any(f.startswith(("20-F", "40-F", "6-K", "20FR", "40FR", "F-")) for f in forms)

    foreign_incorporation = bool(soi) and soi not in US_STATE_CODES
    foreign_address = bool(biz) and biz not in US_STATE_CODES

    notes = []
    if foreign_incorporation:
        notes.append(f"incorporated in {soi_desc or soi}")
    if foreign_address:
        notes.append(f"business address in {biz_desc or biz}")

    return {"sec_company_name": meta.get("name"), "entity_type": meta.get("entityType"),
            "state_of_incorporation": soi or None, "incorporation_desc": soi_desc or None,
            "business_country": biz or None, "business_country_desc": biz_desc or None,
            "has_10k": has_10k, "has_20f_40f": has_20f_40f, "is_fpi": is_fpi,
            "n_10k_years": len(tenk_years),
            "first_10k_fy": min(tenk_years) if tenk_years else None,
            "last_10k_fy": max(tenk_years) if tenk_years else None,
            "foreign_incorporation": foreign_incorporation,
            "foreign_address": foreign_address,
            "domicile_note": "; ".join(notes) or "US"}


def tenk_years_in_older_pages(meta, session=None):
    """
    `filings.recent` holds only the latest ~1,000 filings, so a prolific filer's older
    10-Ks can sit in paged files. Merge those years in before judging a company's history.
    """
    years = set()
    for page in (meta.get("filings", {}) or {}).get("files", []) or []:
        name = page.get("name")
        if not name:
            continue
        _throttle()
        try:
            r = (session or requests).get(f"https://data.sec.gov/submissions/{name}",
                                          headers=SEC_HEADERS, timeout=60)
            if r.status_code == 200:
                years |= tenk_years_from_block(r.json())
        except Exception:
            continue
    return years

In [74]:
def add_foreign_flags(companies, max_workers=4):
    known = companies[companies["cik"].notna()]
    ciks = known["cik"].astype(int).tolist()
    print(f"Fetching SEC metadata for {len(ciks)} CIKs "
          f"(~{len(ciks) * MIN_INTERVAL:.0f}s at {1 / MIN_INTERVAL:.0f} req/s) ...")

    session = requests.Session()
    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        metas = list(pool.map(lambda c: fetch_submission_meta(c, session=session), ciks))

    # Retry empties — these are almost always transient rate limiting.
    retry_idx = [i for i, m in enumerate(metas) if m is None]
    if retry_idx:
        print(f"  retrying {len(retry_idx)} CIKs that returned nothing ...")
        for i in retry_idx:
            metas[i] = fetch_submission_meta(ciks[i], session=session, retries=5)
        still = sum(1 for i in retry_idx if metas[i] is None)
        print(f"  {len(retry_idx) - still} recovered, {still} genuinely missing")

    classified = [classify_issuer(m) for m in metas]

    # The `recent` block truncates hard — Vertex shows 6 years there but has filed 16.
    # So every company with older pages gets them merged, not just ones that look short,
    # otherwise n_10k_years is wrong for most established filers.
    recheck = [i for i, _ in enumerate(classified)
               if (metas[i] or {}).get("filings", {}).get("files")]
    if recheck:
        print(f"  merging older filing pages for {len(recheck)} companies "
              "(the recent block only holds ~1,000 filings) ...")
        recovered = 0
        for i in recheck:
            extra = tenk_years_in_older_pages(metas[i], session=session)
            if not extra:
                continue
            recent_block = (metas[i] or {}).get("filings", {}).get("recent", {})
            merged = tenk_years_from_block(recent_block) | extra
            if len(merged) > classified[i]["n_10k_years"]:
                recovered += 1
            classified[i].update(has_10k=bool(merged), n_10k_years=len(merged),
                                 first_10k_fy=min(merged) if merged else None,
                                 last_10k_fy=max(merged) if merged else None)
        print(f"  {recovered} gained additional years from older pages")

    flags = pd.DataFrame(classified, index=known.index)
    out = companies.join(flags)

    missing = out["cik"].isna()
    out.loc[missing, "domicile_note"] = "ticker not found in SEC ticker map"
    for col in ["has_10k", "has_20f_40f", "is_fpi",
                "foreign_incorporation", "foreign_address"]:
        out[col] = out[col].fillna(False).astype(bool)
    out["n_10k_years"] = out["n_10k_years"].fillna(0).astype(int)
    out["domicile_note"] = out["domicile_note"].fillna("")
    return out


companies = add_foreign_flags(companies)
print("\nDone.")

Fetching SEC metadata for 905 CIKs (~118s at 8 req/s) ...
  merging older filing pages for 185 companies (the recent block only holds ~1,000 filings) ...
  115 gained additional years from older pages

Done.


### The companies being dropped

Split by *why* they fail, so the foreign issuers stay visible as a group and the
short-history cut is separated from the no-10-K-at-all cut.

In [75]:
companies["keeps"] = companies["n_10k_years"] >= MIN_10K_YEARS
dropped = companies[~companies["keeps"]].copy()

def drop_reason(r):
    if not r["has_10k"]:
        if r["has_20f_40f"]:
            return "no 10-K - foreign private issuer (files 20-F/40-F)"
        if r["is_fpi"]:
            return "no 10-K - foreign private issuer (6-K / 20FR / 40FR / F-series)"
        return "no 10-K - recent IPO / shell"
    return f"only {r['n_10k_years']} year(s) of 10-K history"

dropped["drop_reason"] = dropped.apply(drop_reason, axis=1)

print(f"Companies dropped: {len(dropped)}\n")
print("  no 10-K at all                  :", int((~dropped["has_10k"]).sum()))
print(f"  has 10-Ks but < {MIN_10K_YEARS} years       :", int(dropped["has_10k"].sum()))
print(f"  (of all dropped, foreign issuers: {int(dropped['is_fpi'].sum())})")

print("\nYears of history among those dropped for short history:")
print(dropped[dropped["has_10k"]]["n_10k_years"].value_counts().sort_index().to_string())

print("\nBy industry:")
print(dropped["industry"].value_counts().to_string())

Companies dropped: 313

  no 10-K at all                  : 200
  has 10-Ks but < 5 years       : 113
  (of all dropped, foreign issuers: 187)

Years of history among those dropped for short history:
n_10k_years
1    22
2    36
3    32
4    23

By industry:
industry
Biotechnology                       182
Software Infrastructure              73
Internet Content And Information     37
Semiconductors                       21


In [76]:
with pd.option_context("display.max_rows", None, "display.max_colwidth", 55):
    display(dropped[["ticker", "company_name", "industry", "incorporation_desc",
                     "n_10k_years", "first_10k_fy", "last_10k_fy", "drop_reason"]]
            .reset_index(drop=True))

,ticker,company_name,industry,incorporation_desc,n_10k_years,first_10k_fy,last_10k_fy,drop_reason
0,ARGX,ARGENX SE,Biotechnology,Netherlands,0,NaN,NaN,no 10-K - foreign private issuer (files 20-F/40-F)
1,BNTX,BioNTech SE,Biotechnology,Germany,0,NaN,NaN,no 10-K - foreign private issuer (files 20-F/40-F)
2,GMAB,GENMAB A/S,Biotechnology,None,0,NaN,NaN,no 10-K - foreign private issuer (files 20-F/40-F)
3,ASND,Ascendis Pharma A/S,Biotechnology,Denmark,0,NaN,NaN,no 10-K - foreign private issuer (files 20-F/40-F)
4,CAI,"Caris Life Sciences, Inc.",Biotechnology,TX,1,2025.0,2025.0,only 1 year(s) of 10-K history
5,ABVX,Abivax S.A.,Biotechnology,France,0,NaN,NaN,no 10-K - foreign private issuer (files 20-F/40-F)
6,BLTE,"BELITE BIO, INC",Biotechnology,Cayman Islands,0,NaN,NaN,no 10-K - foreign private issuer (files 20-F/40-F)
7,CGON,"CG Oncology, Inc.",Biotechnology,DE,3,2023.0,2025.0,only 3 year(s) of 10-K history
8,MANE,"Veradermics, Inc",Biotechnology,DE,1,2025.0,2025.0,only 1 year(s) of 10-K history
9,PBLS,"Parabilis Medicines, Inc.",Biotechnology,DE,0,NaN,NaN,no 10-K - recent IPO / shell


### Foreign-domiciled companies that are *kept*

These are incorporated or headquartered abroad but file ordinary 10-Ks, so their
Item 1A / 2 / 7 sections are extractable and they stay in the universe.

In [77]:
kept_foreign = companies[companies["keeps"] &
                         (companies["foreign_incorporation"] | companies["foreign_address"])]

print(f"Foreign-domiciled but retained (they file 10-Ks): {len(kept_foreign)}")
with pd.option_context("display.max_rows", None, "display.max_colwidth", 55):
    display(kept_foreign[["ticker", "company_name", "industry", "incorporation_desc",
                          "business_country_desc", "has_10k", "domicile_note"]]
            .reset_index(drop=True))

Foreign-domiciled but retained (they file 10-Ks): 33


,ticker,company_name,industry,incorporation_desc,business_country_desc,has_10k,domicile_note
0,RPRX,Royalty Pharma plc,Biotechnology,United Kingdom,NY,True,incorporated in United Kingdom
1,ROIV,Roivant Sciences Ltd.,Biotechnology,Bermuda,None,True,incorporated in Bermuda
2,JAZZ,Jazz Pharmaceuticals plc,Biotechnology,Ireland,None,True,incorporated in Ireland
3,CRSP,CRISPR Therapeutics AG,Biotechnology,Switzerland,None,True,incorporated in Switzerland
4,DFTX,"Definium Therapeutics, Inc.",Biotechnology,"British Columbia, Canada",NY,True,"incorporated in British Columbia, Canada"
5,ABCL,AbCellera Biologics Inc.,Biotechnology,"British Columbia, Canada","British Columbia, Canada",True,"incorporated in British Columbia, Canada; business ..."
6,XENE,Xenon Pharmaceuticals Inc.,Biotechnology,"British Columbia, Canada",None,True,"incorporated in British Columbia, Canada"
7,ZLAB,Zai Lab Ltd,Biotechnology,Cayman Islands,None,True,incorporated in Cayman Islands
8,AUPH,Aurinia Pharmaceuticals Inc.,Biotechnology,None,"Alberta, Canada",True,"business address in Alberta, Canada"
9,URGN,UroGen Pharma Ltd.,Biotechnology,Israel,Israel,True,incorporated in Israel; business address in Israel


### Drop them and count what is left

In [78]:
universe = companies[companies["keeps"]].reset_index(drop=True)

no_10k = int((~companies["has_10k"]).sum())
short = int((companies["has_10k"] & ~companies["keeps"]).sum())

print(f"Total companies scraped          : {len(companies)}")
print(f"Dropped - no 10-K                : {no_10k}")
print(f"Dropped - < {MIN_10K_YEARS} years of history   : {short}")
print(f"Companies remaining              : {len(universe)}")

print("\nRemaining companies per industry:")
print(universe["industry"].value_counts().to_string())

n_foreign_kept = int((universe["foreign_incorporation"] | universe["foreign_address"]).sum())
print(f"\nOf those {len(universe)}, {n_foreign_kept} are foreign-domiciled 10-K filers "
      "that are deliberately retained.")

print("\nYears of 10-K history in the remaining universe:")
print(universe["n_10k_years"].describe()[["min", "25%", "50%", "75%", "max"]].to_string())
print("\nDistribution:")
print(universe["n_10k_years"].value_counts().sort_index().to_string())

assert universe["has_10k"].all(), "every remaining company must file 10-Ks"
assert (universe["n_10k_years"] >= MIN_10K_YEARS).all(), \
    f"every remaining company must have >= {MIN_10K_YEARS} years of 10-K history"
print(f"\nVerified: all {len(universe)} remaining companies file 10-Ks and have at least "
      f"{MIN_10K_YEARS} fiscal years of history since {START_YEAR}.")

Total companies scraped          : 905
Dropped - no 10-K                : 200
Dropped - < 5 years of history   : 113
Companies remaining              : 592

Remaining companies per industry:
industry
Biotechnology                       406
Software Infrastructure             104
Semiconductors                       49
Internet Content And Information     33

Of those 592, 33 are foreign-domiciled 10-K filers that are deliberately retained.

Years of 10-K history in the remaining universe:
min     5.0
25%     6.0
50%     9.0
75%    15.0
max    16.0

Distribution:
n_10k_years
5     101
6      85
7      43
8      34
9      34
10     26
11     25
12     31
13     25
14     14
15     39
16    135

Verified: all 592 remaining companies file 10-Ks and have at least 5 fiscal years of history since 2011.


In [79]:
with pd.option_context("display.max_rows", None):
    display(universe[["ticker", "company_name", "industry", "cik", "market_cap",
                      "n_10k_years", "first_10k_fy", "last_10k_fy", "domicile_note"]])

,ticker,company_name,industry,cik,market_cap,n_10k_years,first_10k_fy,last_10k_fy,domicile_note
0,VRTX,VERTEX PHARMACEUTICALS INC / MA,Biotechnology,875320,133.37B,16,2010.0,2025.0,US
1,MRNA,"Moderna, Inc.",Biotechnology,1682852,79.40B,8,2018.0,2025.0,US
2,REGN,"REGENERON PHARMACEUTICALS, INC.",Biotechnology,872589,78.69B,16,2010.0,2025.0,US
3,RVMD,"Revolution Medicines, Inc.",Biotechnology,1628171,43.93B,7,2019.0,2025.0,US
4,ONC,BeOne Medicines Ltd.,Biotechnology,1651308,41.03B,11,2015.0,2025.0,US
5,ALNY,"ALNYLAM PHARMACEUTICALS, INC.",Biotechnology,1178670,34.25B,16,2010.0,2025.0,US
6,RPRX,Royalty Pharma plc,Biotechnology,1802768,33.47B,6,2020.0,2025.0,incorporated in United Kingdom
7,ROIV,Roivant Sciences Ltd.,Biotechnology,1635088,25.86B,5,2022.0,2026.0,incorporated in Bermuda
8,INSM,INSMED Inc,Biotechnology,1104506,25.73B,16,2010.0,2025.0,US
9,INCY,INCYTE CORP,Biotechnology,879169,25.11B,16,2010.0,2025.0,US


In [80]:
output_path = '/content/592_companies_universe.csv'

# Select relevant columns from the universe DataFrame
columns_to_save = [
    'ticker',
    'company_name',
    'industry',
    'cik',
    'market_cap',
    'n_10k_years',
    'first_10k_fy',
    'last_10k_fy',
    'domicile_note'
]

universe[columns_to_save].to_csv(output_path, index=False)

print(f"Successfully saved data for {len(universe)} companies to {output_path}")

Successfully saved data for 592 companies to /content/592_companies_universe.csv


## 4. Extract MD&A, Properties and Risk Factors from every 10-K

`edgartools` segments a 10-K into its items, which works well right back to 2011.
Two things still have to be handled, and both showed up in testing:

1. **Transient SEC failures.** A throttled request surfaces as an *empty* item rather than an
   exception, so an all-empty result is retried before it is believed.
2. **`filing.text()` can break.** For some filings edgartools cannot resolve the primary
   document (`'NoneType' object has no attribute 'download'`). `filing.filing_url` still points
   at the right document, so the fallback downloads it directly, strips the HTML and slices the
   items with a regex.

Only exact `10-K` forms are used — `10-K/A` amendments are normally Part III only and would
otherwise create empty duplicate fiscal years.

In [81]:
SECTIONS = [
    ("Item 1A", "Risk Factors", "risk_factors"),
    ("Item 2", "Properties", "properties"),
    ("Item 7", "Management's Discussion and Analysis", "mda"),
]

COLUMN_ORDER = [
    "ticker", "cik", "company_name", "form", "section", "item_label",
    "filing_date", "period_of_report", "fiscal_year", "accession_no",
    "text", "n_chars", "n_words", "source_url",
]


def clean_text(raw):
    if not isinstance(raw, str):
        return ""
    t = (raw.replace("\xa0", " ").replace(" ", " ").replace(" ", " "))
    t = re.sub(r"[ \t]+", " ", t)
    t = re.sub(r"\n\s*\n\s*\n+", "\n\n", t)
    return t.strip()


def html_to_text(html):
    """Strip an EDGAR HTML document down to readable text."""
    soup = BeautifulSoup(html, "lxml")
    for tag in soup(["script", "style"]):
        tag.decompose()
    for td in soup.find_all(["td", "th"]):       # keep table cells apart
        td.insert_after(" ")
    for br in soup.find_all(["br", "p", "div", "tr"]):
        br.insert_after("\n")
    return soup.get_text()


def filing_full_text(filing):
    """Primary-document text, downloading it directly if edgartools cannot."""
    try:
        fn = getattr(filing, "text", None)
        if callable(fn):
            t = fn()
            if t and len(t) > 2000:
                return t
    except Exception:
        pass

    url = getattr(filing, "filing_url", None)
    if not url:
        return ""
    for attempt in range(3):
        _throttle()
        try:
            r = requests.get(url, headers=SEC_HEADERS, timeout=120)
            if r.status_code == 200:
                return r.text if url.lower().endswith(".txt") else html_to_text(r.text)
            time.sleep(1.0 + 2 * attempt)
        except Exception:
            time.sleep(1.0 + 2 * attempt)
    return ""

In [82]:
# Where each item starts, and where the following item begins.
_ITEM_PATTERNS = {
    "Item 1A": r"item\s*1a\.?\s*[-–—:.]?\s*risk\s+factors",
    "Item 2":  r"item\s*2\.?\s*[-–—:.]?\s*propert(?:y|ies)",
    "Item 7":  r"item\s*7\.?\s*[-–—:.]?\s*management",
}
_NEXT_ITEM = {
    "Item 1A": r"item\s*1b\.?\s*[-–—:.]?\s*unresolved|item\s*2\.?\s*[-–—:.]?\s*propert",
    "Item 2":  r"item\s*3\.?\s*[-–—:.]?\s*legal",
    "Item 7":  r"item\s*7a\.?\s*[-–—:.]?\s*quantitative|item\s*8\.?\s*[-–—:.]?\s*financial",
}


def regex_slice(full_text, item_key):
    """Last-resort slicer. The table of contents also matches, so keep the longest slice."""
    if not full_text:
        return ""
    start_pat, end_pat = _ITEM_PATTERNS.get(item_key), _NEXT_ITEM.get(item_key)
    if not start_pat:
        return ""
    best = ""
    for m in re.finditer(start_pat, full_text, re.I):
        s = m.start()
        ends = [e.start() for e in re.finditer(end_pat, full_text[s + 50:], re.I)]
        e = s + 50 + ends[0] if ends else min(len(full_text), s + 400_000)
        chunk = full_text[s:e]
        if len(chunk) > len(best):
            best = chunk
    return best


def _items_from_obj(filing, attempts=3):
    """Ask edgartools for the three items, retrying transient SEC failures."""
    last = {}
    for attempt in range(attempts):
        try:
            obj = filing.obj()
        except Exception:
            time.sleep(1.0 + 2 * attempt)
            continue

        got = {}
        for item_key, _, section in SECTIONS:
            text = ""
            try:
                text = obj[item_key] or ""
            except Exception:
                text = ""
            if not text and section == "mda":
                text = getattr(obj, "management_discussion", "") or ""
            if not text and section == "risk_factors":
                text = getattr(obj, "risk_factors", "") or ""
            got[item_key] = text

        if any(got.values()):
            return got
        last = got                      # all empty -> probably throttled, try again
        time.sleep(1.5 + 2 * attempt)
    return last

In [83]:
def extract_filing_sections(ticker, cik, company_name, filing):
    """One row per section found in a single 10-K filing."""
    rows = []
    items = _items_from_obj(filing)
    full_text = None                    # fetched only if an item lookup came back empty

    period = str(getattr(filing, "period_of_report", "") or "")
    fdate = str(filing.filing_date)
    fiscal_year = int(period[:4]) if period[:4].isdigit() else int(fdate[:4])

    for item_key, item_label, section in SECTIONS:
        text = items.get(item_key, "")
        if not text:
            if full_text is None:
                full_text = filing_full_text(filing)
            text = regex_slice(full_text, item_key)

        text = clean_text(text)
        if not text:
            continue

        rows.append({
            "ticker": ticker,
            "cik": int(cik),
            "company_name": company_name,
            "form": filing.form,
            "section": section,
            "item_label": f"{item_key} - {item_label}",
            "filing_date": fdate,
            "period_of_report": period or None,
            "fiscal_year": fiscal_year,
            "accession_no": filing.accession_no,
            "text": text,
            "n_chars": len(text),
            "n_words": len(text.split()),
            "source_url": getattr(filing, "filing_url", None),
        })
    return rows


def extract_company_sections(ticker, cik, company_name, years=YEARS_BACK, verbose=True):
    start_year = pd.Timestamp.today().year - years
    filings = Company(ticker).get_filings(form="10-K")
    if not filings or len(filings) == 0:
        raise ValueError(f"No 10-K filings found for {ticker}")

    in_window = [f for f in filings if int(str(f.filing_date)[:4]) >= start_year]
    keep = sorted([f for f in in_window if f.form == "10-K"], key=lambda f: str(f.filing_date))
    if verbose:
        n_amend = len(in_window) - len(keep)
        extra = f" ({n_amend} amendment(s) skipped)" if n_amend else ""
        print(f"  {ticker}: {len(keep)} 10-K filings since {start_year}{extra}")

    rows = []
    for f in keep:
        try:
            got = extract_filing_sections(ticker, cik, company_name, f)
            rows.extend(got)
            if verbose:
                found = ", ".join(sorted({g["section"] for g in got})) or "NONE"
                print(f"    {f.filing_date}  FY{str(f.period_of_report)[:4]}:  {found}")
        except Exception as e:
            print(f"    {f.filing_date}: FAILED {type(e).__name__}: {e}")
        time.sleep(0.2)

    if not rows:
        raise ValueError(f"No sections extracted for {ticker}")
    return pd.DataFrame(rows)[COLUMN_ORDER].sort_values(
        ["fiscal_year", "section"]).reset_index(drop=True)

## 5. Pick the first 3 companies

`universe` keeps the market-cap ordering of the source pages, so "the first 3 companies" are
the three largest in the list. Switch to `universe.sort_values("ticker")` first if you would
rather take them alphabetically.

In [84]:
targets = universe

print("Companies selected for this run:")
display(targets[["ticker", "company_name", "industry", "cik", "market_cap"]])

print(f"\nParquet files go straight into: {DROPBOX_RAW}")

Companies selected for this run:


,ticker,company_name,industry,cik,market_cap
0,VRTX,VERTEX PHARMACEUTICALS INC / MA,Biotechnology,875320,133.37B
1,MRNA,"Moderna, Inc.",Biotechnology,1682852,79.40B
2,REGN,"REGENERON PHARMACEUTICALS, INC.",Biotechnology,872589,78.69B
3,RVMD,"Revolution Medicines, Inc.",Biotechnology,1628171,43.93B
4,ONC,BeOne Medicines Ltd.,Biotechnology,1651308,41.03B
...,...,...,...,...,...
587,BLIN,"Bridgeline Digital, Inc.",Software Infrastructure,1378590,10.48M
588,AUID,authID Inc.,Software Infrastructure,1534154,7.76M
589,OLB,"OLB GROUP, INC.",Software Infrastructure,1314196,4.89M
590,YYAI,AIRWA INC.,Software Infrastructure,1674440,3.64M



Parquet files go straight into: /content/bond_shared/data/raw/text


## 6. Extract and store one parquet per company

Each file holds the columns requested: `ticker`, `cik`, `company_name`, `form`, `section`,
`item_label`, `filing_date`, `period_of_report`, `fiscal_year`, `accession_no`, `text`,
`n_chars`, `n_words`, `source_url`.

In [85]:
def store_sections_parquet(df, ticker, raw_path, overwrite=True):
    """One flat file per company: <raw_path>/<TICKER>_10K_sections.parquet"""
    os.makedirs(raw_path, exist_ok=True)
    path = os.path.join(raw_path, f"{ticker}_10K_sections.parquet")
    if os.path.exists(path) and not overwrite:
        print(f"  Skipping (exists): {path}")
        return path
    df.to_parquet(path, index=False, compression="snappy")
    return path


written, failed = [], []
for _, row in targets.iterrows():
    print(f"\n=== {row.ticker} — {row.company_name} (CIK {int(row.cik)}) ===")
    try:
        sections_df = extract_company_sections(row.ticker, int(row.cik), row.company_name)
        path = store_sections_parquet(sections_df, row.ticker, DROPBOX_RAW)
        print(f"  -> {len(sections_df)} rows | {sections_df.n_words.sum():,} words "
              f"| {os.path.getsize(path) / 1e6:.2f} MB")
        print(f"  -> {path}")
        written.append(path)
    except Exception as e:
        print(f"  FAILED: {type(e).__name__}: {e}")
        failed.append({"ticker": row.ticker, "error": str(e)})

print(f"\nStored {len(written)} file(s); {len(failed)} failed.")


=== VRTX — VERTEX PHARMACEUTICALS INC / MA (CIK 875320) ===
  VRTX: 16 10-K filings since 2011
    2011-02-17  FY2010:  mda, properties, risk_factors
    2012-02-22  FY2011:  mda, properties, risk_factors
    2013-03-01  FY2012:  mda, properties, risk_factors
    2014-02-11  FY2013:  mda, properties, risk_factors
    2015-02-13  FY2014:  mda, properties, risk_factors
    2016-02-16  FY2015:  mda, properties, risk_factors
    2017-02-23  FY2016:  mda, properties, risk_factors
    2018-02-15  FY2017:  mda, properties, risk_factors
    2019-02-13  FY2018:  mda, properties, risk_factors
    2020-02-13  FY2019:  mda, properties, risk_factors
    2021-02-11  FY2020:  mda, properties, risk_factors
    2022-02-09  FY2021:  mda, properties, risk_factors
    2023-02-10  FY2022:  mda, properties, risk_factors


    2024-02-15  FY2023:  mda, properties, risk_factors
    2025-02-13  FY2024:  mda, properties, risk_factors
    2026-02-13  FY2025:  mda, properties, risk_factors
  -> 48 rows | 454,484 words | 1.40 MB
  -> /content/bond_shared/data/raw/text/VRTX_10K_sections.parquet

=== MRNA — Moderna, Inc. (CIK 1682852) ===
  MRNA: 8 10-K filings since 2011 (1 amendment(s) skipped)
    2019-03-13  FY2018:  mda, properties, risk_factors
    2020-02-27  FY2019:  mda, properties, risk_factors
    2021-02-26  FY2020:  mda, properties, risk_factors
    2022-02-25  FY2021:  mda, properties, risk_factors
    2023-02-24  FY2022:  mda, properties, risk_factors


    2024-02-23  FY2023:  mda, properties, risk_factors
    2025-02-21  FY2024:  mda, properties, risk_factors
    2026-02-20  FY2025:  mda, properties, risk_factors
  -> 24 rows | 361,654 words | 1.12 MB
  -> /content/bond_shared/data/raw/text/MRNA_10K_sections.parquet

=== REGN — REGENERON PHARMACEUTICALS, INC. (CIK 872589) ===
  REGN: 16 10-K filings since 2011
    2011-02-17  FY2010:  mda, properties, risk_factors
    2012-02-21  FY2011:  mda, properties, risk_factors


    2013-02-15  FY2012:  mda, properties, risk_factors


    2014-02-13  FY2013:  mda, properties, risk_factors


...

    2023-09-28  FY2023:  mda, properties, risk_factors
    2024-09-30  FY2024:  mda, properties, risk_factors
    2025-09-29  FY2025:  mda, properties, risk_factors
  -> 15 rows | 78,624 words | 0.25 MB
  -> /content/bond_shared/data/raw/text/BTLN_10K_sections.parquet

=== CISO — CISO Global, Inc. (CIK 1777319) ===
  CISO: 7 10-K filings since 2011 (2 amendment(s) skipped)
    2020-03-30  FY2019:  mda, properties, risk_factors
    2021-03-31  FY2020:  mda, properties, risk_factors
    2022-04-15  FY2021:  mda, properties, risk_factors
    2023-03-31  FY2022:  mda, properties, risk_factors
    2024-04-16  FY2023:  mda, properties, risk_factors
    2025-03-31  FY2024:  mda, properties, risk_factors
    2026-03-30  FY2025:  mda, properties, risk_factors
  -> 21 rows | 99,107 words | 0.31 MB
  -> /content/bond_shared/data/raw/text/CISO_10K_sections.parquet

=== BLIN — Bridgeline Digital, Inc. (CIK 1378590) ===
  BLIN: 15 10-K filings since 2011 (5 amendment(s) skipped)
    2011-12-29  FY20

    2012-03-28  FY2011:  mda, properties


    2013-04-01  FY2012:  mda, properties
    2014-04-15  FY2013:  mda
    2015-03-31  FY2014:  mda, properties, risk_factors
    2016-11-02  FY2015:  mda, properties, risk_factors
    2017-07-12  FY2016:  mda, properties, risk_factors
    2018-03-15  FY2017:  mda, properties, risk_factors
    2019-03-08  FY2018:  mda, properties, risk_factors
    2020-03-30  FY2019:  mda, properties, risk_factors
    2021-03-08  FY2020:  mda, properties, risk_factors
    2022-03-22  FY2021:  mda, properties, risk_factors
    2023-03-30  FY2022:  mda, properties, risk_factors
    2024-03-20  FY2023:  mda, properties, risk_factors
    2025-03-13  FY2024:  mda, properties, risk_factors
    2026-03-31  FY2025:  mda, properties, risk_factors
  -> 41 rows | 179,953 words | 0.57 MB
  -> /content/bond_shared/data/raw/text/AUID_10K_sections.parquet

=== OLB — OLB GROUP, INC. (CIK 1314196) ===
  OLB: 16 10-K filings since 2011 (3 amendment(s) skipped)
    2011-03-25  FY2010:  mda, properties, risk_factors
    20

    2023-09-14  FY2023:  mda, properties, risk_factors
    2024-07-25  FY2024:  mda, properties, risk_factors
    2025-08-13  FY2025:  mda, properties, risk_factors
    2026-09-21  FY2026:  mda, properties, risk_factors
  -> 30 rows | 177,400 words | 0.54 MB
  -> /content/bond_shared/data/raw/text/YYAI_10K_sections.parquet

=== CETX — CEMTREX INC (CIK 1435064) ===
  CETX: 16 10-K filings since 2011 (17 amendment(s) skipped)
    2011-01-14  FY2010:  mda, properties, risk_factors
    2012-01-10  FY2011:  mda, properties, risk_factors
    2012-12-26  FY2012:  mda, properties, risk_factors
    2014-01-16  FY2013:  mda, properties, risk_factors
    2014-12-29  FY2014:  mda, properties, risk_factors
    2015-12-21  FY2015:  mda, properties, risk_factors
    2016-12-28  FY2016:  mda, properties, risk_factors
    2017-12-13  FY2017:  mda, properties, risk_factors
    2019-01-11  FY2018:  mda, properties, risk_factors
    2020-01-14  FY2019:  mda, properties, risk_factors
    2021-01-05  FY2020

    2022-01-21  FY2021:  mda, properties, risk_factors
    2022-12-28  FY2022:  mda, properties, risk_factors
    2023-12-28  FY2023:  mda, properties, risk_factors
    2024-12-30  FY2024:  mda, properties, risk_factors
    2025-12-29  FY2025:  mda, properties, risk_factors
  -> 48 rows | 150,150 words | 0.45 MB
  -> /content/bond_shared/data/raw/text/CETX_10K_sections.parquet

Stored 592 file(s); 0 failed.


## 7. Read the parquet files back

In [86]:
import glob

files = sorted(glob.glob(os.path.join(DROPBOX_RAW, "*_10K_sections.parquet")))
print(f"Parquet files in {DROPBOX_RAW}:")
for f in files:
    print(f"  {os.path.basename(f)}  ({os.path.getsize(f) / 1e6:.2f} MB)")

sections = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)
print(f"\nCombined shape: {sections.shape}")
print("Columns:", list(sections.columns))

Parquet files in /content/bond_shared/data/raw/text:
  ABCL_10K_sections.parquet  (0.87 MB)
  ABEO_10K_sections.parquet  (0.92 MB)
  ABOS_10K_sections.parquet  (0.71 MB)
  ABSI_10K_sections.parquet  (0.66 MB)
  ABUS_10K_sections.parquet  (0.89 MB)
  ABVC_10K_sections.parquet  (0.89 MB)
  ACAD_10K_sections.parquet  (1.36 MB)
  ACET_10K_sections.parquet  (1.25 MB)
  ACHV_10K_sections.parquet  (1.27 MB)
  ACIW_10K_sections.parquet  (1.00 MB)
  ACRS_10K_sections.parquet  (1.19 MB)
  ACXP_10K_sections.parquet  (0.44 MB)
  ADIL_10K_sections.parquet  (0.82 MB)
  ADI_10K_sections.parquet  (0.84 MB)
  ADMA_10K_sections.parquet  (1.21 MB)
  AEON_10K_sections.parquet  (0.65 MB)
  AEVA_10K_sections.parquet  (0.46 MB)
  AGEN_10K_sections.parquet  (1.73 MB)
  AGIO_10K_sections.parquet  (1.34 MB)
  AGPU_10K_sections.parquet  (0.75 MB)
  AIFA_10K_sections.parquet  (0.45 MB)
  AIFC_10K_sections.parquet  (0.67 MB)
  AIM_10K_sections.parquet  (0.74 MB)
  AIP_10K_sections.parquet  (0.56 MB)
  AISP_10K_sec

### Verify actual files in `DROPBOX_RAW`

In [87]:
import os

print(f"Listing files in the actual DROPBOX_RAW directory: {DROPBOX_RAW}")
for root, dirs, files in os.walk(DROPBOX_RAW):
    for file in files:
        print(os.path.join(root, file))

Listing files in the actual DROPBOX_RAW directory: /content/bond_shared/data/raw/text
/content/bond_shared/data/raw/text/SNDX_10K_sections.parquet
/content/bond_shared/data/raw/text/IIIV_10K_sections.parquet
/content/bond_shared/data/raw/text/MPWR_10K_sections.parquet
/content/bond_shared/data/raw/text/LPCN_10K_sections.parquet
/content/bond_shared/data/raw/text/URGN_10K_sections.parquet
/content/bond_shared/data/raw/text/PDSB_10K_sections.parquet
/content/bond_shared/data/raw/text/CDXS_10K_sections.parquet
/content/bond_shared/data/raw/text/YEXT_10K_sections.parquet
/content/bond_shared/data/raw/text/AUID_10K_sections.parquet
/content/bond_shared/data/raw/text/CVM_10K_sections.parquet
/content/bond_shared/data/raw/text/KTTA_10K_sections.parquet
/content/bond_shared/data/raw/text/RPAY_10K_sections.parquet
/content/bond_shared/data/raw/text/QTTB_10K_sections.parquet
/content/bond_shared/data/raw/text/DTIL_10K_sections.parquet
/content/bond_shared/data/raw/text/ABVC_10K_sections.parquet


In [88]:
print("Rows per company and section:")
print(pd.crosstab(sections["ticker"], sections["section"]).to_string())

print("\nFiscal-year coverage:")
for t, g in sections.groupby("ticker"):
    print(f"  {t}: FY{g.fiscal_year.min()}-FY{g.fiscal_year.max()} "
          f"({g.fiscal_year.nunique()} years, {g.accession_no.nunique()} filings)")

print("\nMissing values per column:")
print(sections.isna().sum().to_string())

print("\nWord counts by section:")
print(sections.groupby("section")["n_words"].agg(["min", "median", "max", "sum"]).to_string())

Rows per company and section:
section  mda  properties  risk_factors
ticker                                
ABCL       6           6             6
ABEO      16          16            16
ABOS       5           5             5
ABSI       5           5             5
ABUS      13          13            13
ABVC      16          16            16
ACAD      16          16            16
ACET       9           9             9
ACHV      16          16            16
ACIW      16          16            16
ACRS      11          11            11
ACXP       5           5             5
ADI       15          15            15
ADIL       8           8             8
ADMA      15          15            15
AEON       6           6             6
AEVA       7           7             7
AGEN      16          16            16
AGIO      13          13            13
AGPU      16          16            16
AI         6           6             6
AIFA       9           9             9
AIFC      16          16          

In [89]:
# Metadata view (text column dropped so the table is readable).
display(sections.drop(columns=["text"]).head(12))

,ticker,cik,company_name,form,section,item_label,filing_date,period_of_report,fiscal_year,accession_no,n_chars,n_words,source_url
0,ABCL,1703057,AbCellera Biologics Inc.,10-K,mda,Item 7 - Management's Discussion and Analysis,2021-03-30,2020-12-31,2020,0001564590-21-016713,91244,13991,https://www.sec.gov/Archives/edgar/data/1703057/00015645...
1,ABCL,1703057,AbCellera Biologics Inc.,10-K,properties,Item 2 - Properties,2021-03-30,2020-12-31,2020,0001564590-21-016713,796,122,https://www.sec.gov/Archives/edgar/data/1703057/00015645...
2,ABCL,1703057,AbCellera Biologics Inc.,10-K,risk_factors,Item 1A - Risk Factors,2021-03-30,2020-12-31,2020,0001564590-21-016713,208177,31786,https://www.sec.gov/Archives/edgar/data/1703057/00015645...
3,ABCL,1703057,AbCellera Biologics Inc.,10-K,mda,Item 7 - Management's Discussion and Analysis,2022-02-25,2021-12-31,2021,0001564590-22-007077,79885,12074,https://www.sec.gov/Archives/edgar/data/1703057/00015645...
4,ABCL,1703057,AbCellera Biologics Inc.,10-K,properties,Item 2 - Properties,2022-02-25,2021-12-31,2021,0001564590-22-007077,1045,162,https://www.sec.gov/Archives/edgar/data/1703057/00015645...
5,ABCL,1703057,AbCellera Biologics Inc.,10-K,risk_factors,Item 1A - Risk Factors,2022-02-25,2021-12-31,2021,0001564590-22-007077,215569,32823,https://www.sec.gov/Archives/edgar/data/1703057/00015645...
6,ABCL,1703057,AbCellera Biologics Inc.,10-K,mda,Item 7 - Management's Discussion and Analysis,2023-02-21,2022-12-31,2022,0001564590-23-002128,74525,11219,https://www.sec.gov/Archives/edgar/data/1703057/00015645...
7,ABCL,1703057,AbCellera Biologics Inc.,10-K,properties,Item 2 - Properties,2023-02-21,2022-12-31,2022,0001564590-23-002128,1098,170,https://www.sec.gov/Archives/edgar/data/1703057/00015645...
8,ABCL,1703057,AbCellera Biologics Inc.,10-K,risk_factors,Item 1A - Risk Factors,2023-02-21,2022-12-31,2022,0001564590-23-002128,236418,35754,https://www.sec.gov/Archives/edgar/data/1703057/00015645...
9,ABCL,1703057,AbCellera Biologics Inc.,10-K,mda,Item 7 - Management's Discussion and Analysis,2024-02-20,2023-12-31,2023,0001703057-24-000019,67047,9902,https://www.sec.gov/Archives/edgar/data/1703057/00017030...


In [90]:
# Spot-check the actual text that was captured.
for section in ["mda", "properties", "risk_factors"]:
    row = sections[(sections.ticker == sections.ticker.iloc[0]) &
                   (sections.section == section)].iloc[-1]
    print("=" * 100)
    print(f"{row.ticker}  FY{row.fiscal_year}  {row.item_label}  "
          f"({row.n_words:,} words, {row.n_chars:,} chars)")
    print(f"{row.source_url}")
    print("-" * 100)
    print(row.text[:700].strip())
    print("   ... [middle omitted] ...")
    print(row.text[-400:].strip())
    print()

ABCL  FY2025  Item 7 - Management's Discussion and Analysis  (7,580 words, 52,093 chars)
https://www.sec.gov/Archives/edgar/data/1703057/000170305726000012/abcl-20251231.htm
----------------------------------------------------------------------------------------------------
Item 7. Management’s Discussion and Analysis of Financial Condition and Results of Operations. 

You should read the following discussion and analysis of our financial condition and results of operations together with our consolidated financial statements and the related notes thereto included elsewhere in this annual report. Some of the information contained in this discussion and analysis or set forth in other parts of this annual report contain forward-looking statements that involve risks, uncertainties and assumptions. As a result of many factors, including those factors set forth in Part I, Item 1A, Risk Factors, our actual results could differ materially from those discussed in or imp
   ... [middle omitted] 

### Boundary check

A section that over-ran its item would contain the *body heading* of a later item. This
confirms each slice stops where it should.

In [91]:
BLEED = {
    "risk_factors": r"item\s*8\.?\s*[-–—:.]?\s*financial\s+statements\s+and\s+supplementary",
    "properties":   r"item\s*7\.?\s*[-–—:.]?\s*management.s\s+discussion",
    "mda":          r"item\s*9a\.?\s*[-–—:.]?\s*controls\s+and\s+procedures",
}

flagged = 0
for _, r in sections.iterrows():
    hits = [m.start() for m in re.finditer(BLEED[r.section], r.text, re.I) if m.start() > 2000]
    if hits:
        flagged += 1
        print(f"  over-run: {r.ticker} FY{r.fiscal_year} {r.section} at {hits[:2]} of {r.n_chars}")

print("All section boundaries clean." if not flagged else f"{flagged} row(s) flagged.")

  over-run: BBLG FY2017 mda at [4963] of 28198
  over-run: BMBL FY2022 risk_factors at [152051] of 210595
  over-run: BMBL FY2023 risk_factors at [137881] of 185923
  over-run: BMBL FY2024 risk_factors at [143851, 192740] of 194044
  over-run: BMBL FY2025 risk_factors at [181989] of 182514
  over-run: HRMY FY2025 risk_factors at [200619] of 234123
  over-run: IGC FY2011 mda at [2708] of 45817
  over-run: INTZ FY2010 mda at [26512] of 26544
  over-run: INTZ FY2011 mda at [26832] of 26864
  over-run: INTZ FY2012 mda at [27519] of 27551
  over-run: INTZ FY2013 mda at [27457] of 27489
  over-run: INTZ FY2014 mda at [26927] of 26959
  over-run: INTZ FY2015 mda at [26469] of 26501
  over-run: MAX FY2024 risk_factors at [12099] of 161377
  over-run: MAX FY2025 risk_factors at [14042] of 156400
  over-run: MDXG FY2021 risk_factors at [105648] of 128894
  over-run: MU FY2016 risk_factors at [16726, 22517] of 57292
  over-run: MU FY2017 risk_factors at [27779] of 58890
  over-run: MU FY2018 risk

In [93]:
import shutil

# Define the destination folder in your Google Drive
# You can change '10K_sections' to any folder name you prefer in your Drive
drive_output_folder = '/content/drive/MyDrive/10K_sections'

# Create the destination folder in Google Drive if it doesn't exist
os.makedirs(drive_output_folder, exist_ok=True)

print(f"Copying files from {DROPBOX_RAW} to {drive_output_folder}...")

# Copy each parquet file from the source directory to the Google Drive destination
for file_name in os.listdir(DROPBOX_RAW):
    if file_name.endswith('.parquet'):
        source_file_path = os.path.join(DROPBOX_RAW, file_name)
        destination_file_path = os.path.join(drive_output_folder, file_name)
        shutil.copy2(source_file_path, destination_file_path)
        print(f"Copied: {file_name}")

print("All parquet files copied to Google Drive.")

Copying files from /content/bond_shared/data/raw/text to /content/drive/MyDrive/10K_sections...
Copied: SNDX_10K_sections.parquet
Copied: IIIV_10K_sections.parquet
Copied: MPWR_10K_sections.parquet
Copied: LPCN_10K_sections.parquet
Copied: URGN_10K_sections.parquet
Copied: PDSB_10K_sections.parquet
Copied: CDXS_10K_sections.parquet
Copied: YEXT_10K_sections.parquet
Copied: AUID_10K_sections.parquet
Copied: CVM_10K_sections.parquet
Copied: KTTA_10K_sections.parquet
Copied: RPAY_10K_sections.parquet
Copied: QTTB_10K_sections.parquet
Copied: DTIL_10K_sections.parquet
Copied: ABVC_10K_sections.parquet
Copied: GLUE_10K_sections.parquet
Copied: CYTK_10K_sections.parquet
Copied: TENB_10K_sections.parquet
Copied: PHAT_10K_sections.parquet
Copied: VIR_10K_sections.parquet
Copied: APRE_10K_sections.parquet
Copied: AIP_10K_sections.parquet
Copied: ADMA_10K_sections.parquet
Copied: VANI_10K_sections.parquet
Copied: LRMR_10K_sections.parquet
Copied: ANVS_10K_sections.parquet
Copied: MBIO_10K_sectio

In [94]:
import pandas as pd
import os

vrtx_file_path = os.path.join(DROPBOX_RAW, "VRTX_10K_sections.parquet")

if os.path.exists(vrtx_file_path):
    print(f"Reading {vrtx_file_path}...")
    vrtx_df = pd.read_parquet(vrtx_file_path)
    # Display all rows of the DataFrame
    with pd.option_context('display.max_rows', None, 'display.max_columns', None):
        display(vrtx_df)
else:
    print(f"Error: The file {vrtx_file_path} does not exist. Please ensure the extraction process for VRTX was completed successfully.")

Reading /content/bond_shared/data/raw/text/VRTX_10K_sections.parquet...


,ticker,cik,company_name,form,section,item_label,filing_date,period_of_report,fiscal_year,accession_no,text,n_chars,n_words,source_url
0,VRTX,875320,VERTEX PHARMACEUTICALS INC / MA,10-K,mda,Item 7 - Management's Discussion and Analysis,2011-02-17,2010-12-31,2010,0001047469-11-001025,ITEM 7. MANAGEMENT'S DISCUSSION AND ANALYSIS OF FINANCIA...,80486,12244,https://www.sec.gov/Archives/edgar/data/875320/000104746...
1,VRTX,875320,VERTEX PHARMACEUTICALS INC / MA,10-K,properties,Item 2 - Properties,2011-02-17,2010-12-31,2010,0001047469-11-001025,ITEM 2. PROPERTIES \n\n We lease an aggregate of approxi...,3351,529,https://www.sec.gov/Archives/edgar/data/875320/000104746...
2,VRTX,875320,VERTEX PHARMACEUTICALS INC / MA,10-K,risk_factors,Item 1A - Risk Factors,2011-02-17,2010-12-31,2010,0001047469-11-001025,ITEM 1A. RISK FACTORS \n\n RISK FACTORS \n\n Investing i...,81291,12252,https://www.sec.gov/Archives/edgar/data/875320/000104746...
3,VRTX,875320,VERTEX PHARMACEUTICALS INC / MA,10-K,mda,Item 7 - Management's Discussion and Analysis,2012-02-22,2011-12-31,2011,0001047469-12-001298,ITEM 7. MANAGEMENT'S DISCUSSION AND ANALYSIS OF FINANCIA...,79821,12049,https://www.sec.gov/Archives/edgar/data/875320/000104746...
4,VRTX,875320,VERTEX PHARMACEUTICALS INC / MA,10-K,properties,Item 2 - Properties,2012-02-22,2011-12-31,2011,0001047469-12-001298,ITEM 2. PROPERTIES \n\n We lease an aggregate of approxi...,3323,524,https://www.sec.gov/Archives/edgar/data/875320/000104746...
5,VRTX,875320,VERTEX PHARMACEUTICALS INC / MA,10-K,risk_factors,Item 1A - Risk Factors,2012-02-22,2011-12-31,2011,0001047469-12-001298,ITEM 1A. RISK FACTORS \n\n RISK FACTORS \n\n Investing i...,80352,12040,https://www.sec.gov/Archives/edgar/data/875320/000104746...
6,VRTX,875320,VERTEX PHARMACEUTICALS INC / MA,10-K,mda,Item 7 - Management's Discussion and Analysis,2013-03-01,2012-12-31,2012,0000875320-13-000009,Table of Contents\n\nITEM 7. MANAGEMENT'S DISCUSSION AND...,87067,13301,https://www.sec.gov/Archives/edgar/data/875320/000087532...
7,VRTX,875320,VERTEX PHARMACEUTICALS INC / MA,10-K,properties,Item 2 - Properties,2013-03-01,2012-12-31,2012,0000875320-13-000009,ITEM 2. PROPERTIES\n\nMassachusetts Headquarters\n\nWe l...,3862,598,https://www.sec.gov/Archives/edgar/data/875320/000087532...
8,VRTX,875320,VERTEX PHARMACEUTICALS INC / MA,10-K,risk_factors,Item 1A - Risk Factors,2013-03-01,2012-12-31,2012,0000875320-13-000009,Table of Contents\n\nITEM 1A. RISK FACTORS\n\nRISK FACTO...,91892,13829,https://www.sec.gov/Archives/edgar/data/875320/000087532...
9,VRTX,875320,VERTEX PHARMACEUTICALS INC / MA,10-K,mda,Item 7 - Management's Discussion and Analysis,2014-02-11,2013-12-31,2013,0000875320-14-000015,ITEM 7. MANAGEMENT'S DISCUSSION AND ANALYSIS OF FINANCIA...,76453,11598,https://www.sec.gov/Archives/edgar/data/875320/000087532...


## 8. Running the whole universe

Everything above is already written for the full list — this run only stopped at
`N_COMPANIES = 3`. To process every remaining company:

```python
N_COMPANIES = len(universe)
targets = universe                 # then re-run sections 5 and 6
```

Practical notes before you do:

- **Time.** Roughly 5–15 seconds per filing, ~10 filings per company. Every company in
  `universe` now has at least one 10-K, so budget several hours for the full run. It is safe
  to stop and restart — set
  `overwrite=False` in `store_sections_parquet` and completed companies are skipped.
- **Size.** These three companies produced ~1.4 MB of parquet each. The full universe will run
  to roughly 1–2 GB, so let Dropbox finish syncing.
- **Rate limits.** Every SEC call in this notebook goes through `_throttle()`, so leave
  `MIN_INTERVAL` at 0.13 and do not raise `max_workers`.
- **Failures.** Wrap the loop in a `failed` list (as in section 6) and retry those tickers
  afterwards — most failures are transient.


```
bond_shared/data/raw/
├── financials/{TICKER}/10K/{year}_{income|balance|cashflow|equity}.parquet   # XBRL
└── text/{TICKER}_10K_sections.parquet                                        # this notebook
```

Because every file is one flat `{TICKER}_10K_sections.parquet`, the whole text corpus loads
in one line once the full run is done:

```python
corpus = pd.read_parquet(DROPBOX_RAW)     # pyarrow reads the directory as one dataset
```